In [ ]:
import pandas as pd

In [ ]:
drug_df = pd.read_pickle("../../data/outcomes_squashed/outcomes_squashed_cui.pkl")

In [ ]:
drug_df.vector

In [ ]:
from itertools import combinations


def top_cooccurrence_drug_overdoses(df, drug_cols, top_n=20, max_combination_length=5):
    """
    Calculates the top N co-occurring drug overdoses in the dataset for combinations
    up to the specified maximum length.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the drug overdose data.
    drug_cols (list): List of drug columns to analyze for co-occurrence.
    top_n (int): The number of top co-occurrences to return.
    max_combination_length (int): The maximum number of drugs in a combination.

    Returns:
    pd.DataFrame: A DataFrame with the top N co-occurrences and their counts.
    """
    co_occurrence_counts = {}

    # Loop through each combination length (from 2 up to max_combination_length)
    for combination_length in range(2, max_combination_length + 1):
        # Generate all combinations of the specified length
        for drug_combo in combinations(drug_cols, combination_length):
            # Count cases where all drugs in the combination are present (i.e., all are 1)
            count = df[list(drug_combo)].all(axis=1).sum()
            if count > 0:
                co_occurrence_counts[drug_combo] = count

    # Convert to DataFrame for easy sorting and selection
    co_occurrence_df = pd.DataFrame(
        [(combo, count) for combo, count in co_occurrence_counts.items()],
        columns=["Drug Combination", "Count"],
    )

    # Sort by count and get the top N combinations
    top_co_occurrences = co_occurrence_df.nlargest(top_n, "Count")

    return top_co_occurrences

In [ ]:
drug_cols = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Benzodiazepines",
    "Others",
]

# Assuming your DataFrame is named df
top_co_occurrences = top_cooccurrence_drug_overdoses(drug_df, drug_cols)
print(top_co_occurrences)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming you have already run the top_exact_cooccurrence_drug_overdoses function
# and obtained the top_co_occurrences DataFrame


def plot_top_drug_combinations(top_co_occurrences):
    """
    Plots a bar chart of the top drug combinations based on their counts.

    Parameters:
    top_co_occurrences (pd.DataFrame): DataFrame containing 'Drug Combination' and 'Count' columns.
    """
    # Convert the drug combinations from tuples to strings for labeling
    top_co_occurrences = top_co_occurrences.copy()
    top_co_occurrences["Drug Combination"] = top_co_occurrences[
        "Drug Combination"
    ].apply(lambda combo: ", ".join(combo))

    # Set the plot style
    sns.set_style("whitegrid")

    # Create a bar plot
    plt.figure(figsize=(12, 8))
    sns.barplot(
        data=top_co_occurrences, x="Count", y="Drug Combination", palette="viridis"
    )

    # Set plot labels and title
    plt.xlabel("Number of Overdoses")
    plt.ylabel("Drug Combination")
    plt.title("Top 20 Drug Combinations in Overdoses")

    # Adjust layout to fit labels
    plt.tight_layout()

    # Show the plot
    plt.show()

In [ ]:
plot_top_drug_combinations(top_co_occurrences)

In [ ]:
drug_df["Others"].value_counts()